# FEDS: Federated Learning with Adaptive Sparsification

Run FEDS server + MNIST clients in Google Colab.

**What this notebook does:**
1. Clones the FEDS repository
2. Installs dependencies
3. Creates initial model artifacts (MNIST CNN + K values)
4. Starts the FEDS server in background
5. Launches 10 MNIST clients with non-IID data
6. Displays FEDS K statistics and TensorBoard results

**Tip:** For a quick test, interrupt the server cell after a few rounds (Ctrl+C or stop button).

In [ ]:
# Clone repo and set up paths
import os
import sys

# Clone the repository (skip if already cloned)
REPO_NAME = "feds"
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/masud1901/feds.git

REPO = os.path.abspath(REPO_NAME)
os.chdir(REPO)

# Add repo and utils folder to Python path
# This is needed because server.py imports 'from dsfl_feds import ...' (not 'from utils.dsfl_feds')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "utils"))

print(f"Working directory: {os.getcwd()}")
print(f"REPO path: {REPO}")

In [ ]:
# Install dependencies
!pip install -q flwr torch torchvision torchaudio numpy tensorboard matplotlib scipy

import flwr
import torch
print(f"flwr: {flwr.__version__}")
print(f"torch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU for faster training.")

In [ ]:
# Create initial model and K pickle if missing
import subprocess
import sys
import os

# Ensure we're in the right directory
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "utils"))

script_path = os.path.join(REPO, "scripts", "create_artifacts_colab.py")

if os.path.exists(script_path):
    result = subprocess.run(
        [sys.executable, script_path],
        cwd=REPO,
        capture_output=True,
        text=True
    )
    print(result.stdout if result.stdout else "")
    if result.stderr:
        print("Errors:", result.stderr)
else:
    print(f"Script not found: {script_path}")
    print("Creating artifacts inline...")
    
    # Create artifacts inline if script doesn't exist
    import pickle
    import torch.nn as nn
    import torch.nn.functional as F
    
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 32, 5)
            self.pool1 = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(32, 64, 5)
            self.pool2 = nn.MaxPool2d(2, 2)
            self.fc1 = nn.Linear(64 * 4 * 4, 512)
            self.fc2 = nn.Linear(512, 10)

        def forward(self, x):
            x = self.pool1(F.relu(self.conv1(x)))
            x = self.pool2(F.relu(self.conv2(x)))
            x = x.view(-1, 64 * 4 * 4)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return x
    
    if not os.path.exists("initial_global_model_MNIST"):
        net = Net()
        params = [val.cpu().numpy() for _, val in net.state_dict().items()]
        initial_model = [(params, 1)]
        with open("initial_global_model_MNIST", "wb") as f:
            pickle.dump(initial_model, f)
        print("Created initial_global_model_MNIST")
    else:
        print("initial_global_model_MNIST already exists")
    
    k_pickle = "K_alpha=10_gamma=10_test=0.pickle"
    if not os.path.exists(k_pickle):
        d = 582026  # Total MNIST CNN parameters
        k_list = [d // 2] * 10  # 50% sparsity for 10 clients
        with open(k_pickle, "wb") as f:
            pickle.dump(k_list, f)
        print(f"Created {k_pickle}")
    else:
        print(f"{k_pickle} already exists")

In [ ]:
# Run FEDS server in background
import subprocess
import time
import os

# Set environment with proper Python paths
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO}:{os.path.join(REPO, 'utils')}"

# Clean up old FEDS files
for f in ['feds_k_tracker.json', 'feds_history.json', 'feds_k_trajectory.json']:
    if os.path.exists(f):
        os.remove(f)

server_script = os.path.join(REPO, "server.py")
print(f"Starting server: {server_script}")
print("=" * 60)

# Run server in background - output goes to notebook cell
server_proc = subprocess.Popen(
    [sys.executable, server_script],
    cwd=REPO,
    env=env
)

# Wait for server to initialize
time.sleep(10)
print("Server started! Waiting for clients to connect...")

In [ ]:
# Run 10 MNIST clients (non-IID data distribution)
import subprocess
import os

client_script = os.path.join(REPO, "clients", "client-MNIST.py")
print(f"Starting 10 clients from: {client_script}")
print("=" * 60)

# Set environment with proper Python paths
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO}:{os.path.join(REPO, 'utils')}"

procs = []
for i in range(10):
    p = subprocess.Popen(
        [sys.executable, client_script, "--seed", str(i)],
        cwd=REPO,
        env=env,
        stdout=subprocess.DEVNULL,  # Suppress client output (server shows the important stats)
        stderr=subprocess.DEVNULL
    )
    procs.append(p)
    print(f"Started client {i} (PID: {p.pid})")

print("\nWaiting for all clients to finish training...")
for i, p in enumerate(procs):
    p.wait()
    print(f"Client {i} finished.")

print("\n" + "=" * 60)
print("All clients finished!")

In [ ]:
# Check FEDS outputs and results
import json
import os

print("=" * 60)
print("FEDS EXPERIMENT RESULTS")
print("=" * 60)

# Display K tracker stats
k_tracker_file = os.path.join(REPO, "feds_k_tracker.json")
if os.path.exists(k_tracker_file):
    with open(k_tracker_file) as f:
        k_data = json.load(f)
        k_list = k_data.get('k_list', [])
        loss_history = k_data.get('loss_history', [])
        
        print(f"\nK values (sparsity control):")
        print(f"  Mean K: {sum(k_list)/len(k_list):.0f}")
        print(f"  Min K: {min(k_list)}, Max K: {max(k_list)}")
        print(f"  K values per client: {k_list}")
        
        if loss_history:
            latest_losses = loss_history[-1]
            print(f"\nLatest round losses:")
            print(f"  Mean loss: {sum(latest_losses)/len(latest_losses):.4f}")
            print(f"  Min loss: {min(latest_losses):.4f}, Max loss: {max(latest_losses):.4f}")
else:
    print("\nNo feds_k_tracker.json found - training may not have completed.")

# Display K trajectory
trajectory_file = os.path.join(REPO, "feds_k_trajectory.json")
if os.path.exists(trajectory_file):
    with open(trajectory_file) as f:
        traj_data = json.load(f)
        k_history = traj_data.get('k_history', [])
        print(f"\nK trajectory length: {len(k_history)} rounds")
else:
    print("\nNo feds_k_trajectory.json found.")

# Check TensorBoard runs
runs_dir = os.path.join(REPO, "runs")
if os.path.isdir(runs_dir) and os.listdir(runs_dir):
    print(f"\nTensorBoard logs available:")
    for run in os.listdir(runs_dir):
        print(f"  - {run}")
    print(f"\nTo view TensorBoard logs, download the 'runs' folder and run:")
    print(f"  tensorboard --logdir=runs")
else:
    print("\nNo TensorBoard runs directory found.")

print("\n" + "=" * 60)
print("Done!")

In [ ]:
# Stop the server
server_proc.terminate()
server_proc.wait()
print("Server stopped.")